In [1]:
import os
import sqlite3
import json
import base64
import requests
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from pathlib import Path

DB_PATH = "kneevision_final.db"
DATASET_DIR = "../dataset_preprocessed"
MODEL_PATH = "best_model_b4.pt"
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2-vision:11b"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
MAX_PER_GRADE = 25

GRADE_FOLDERS = {
    "Grade_0": 0,
    "Grade_1": 1,
    "Grade_2": 2,
    "Grade_3": 3,
    "Grade_4": 4,
}

def init_db(conn):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS cases (
            case_id TEXT PRIMARY KEY,
            image_path TEXT NOT NULL,
            kl_grade INTEGER NOT NULL,
            dataset_source TEXT,
            osteophyte_severity TEXT,
            joint_space_narrowing TEXT,
            subchondral_sclerosis TEXT,
            bone_texture TEXT,
            affected_compartment TEXT,
            overall_findings TEXT,
            raw_metadata TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS features (
            case_id TEXT PRIMARY KEY,
            feature_vector BLOB NOT NULL,
            FOREIGN KEY (case_id) REFERENCES cases(case_id)
        )
    """)
    conn.commit()

def load_feature_extractor():
    model = models.efficientnet_b4(weights=None)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    state = checkpoint.get("model_state_dict", checkpoint)
    
    # strip backbone. prefix, skip classifier
    new_state = {}
    for k, v in state.items():
        if k.startswith("backbone.classifier"):
            continue
        if k.startswith("backbone."):
            new_state[k[len("backbone."):]] = v
    
    model.load_state_dict(new_state, strict=False)
    
    # replace classifier with identity to get feature vectors
    model.classifier = nn.Identity()
    model.eval()
    model.to(DEVICE)
    return model

def extract_features(model, image_path):
    transform = transforms.Compose([
        transforms.Resize((380, 380)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        vec = model(tensor).squeeze().cpu().numpy()
    return vec

def image_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def get_metadata_from_ollama(image_path, kl_grade):
    prompt = f"""You are a radiologist analyzing a knee X-ray classified as Kellgren-Lawrence Grade {kl_grade}.

Analyze the image and respond ONLY with a valid JSON object with exactly these fields:
{{
  "osteophyte_severity": "none | mild | moderate | severe",
  "joint_space_narrowing": "none | mild | moderate | severe",
  "subchondral_sclerosis": "none | mild | moderate | severe",
  "bone_texture": "normal | slightly irregular | irregular | severely irregular",
  "affected_compartment": "none | medial | lateral | both | whole joint",
  "overall_findings": "2-3 sentence clinical summary of what is visible in this X-ray"
}}

Do not include any explanation or text outside the JSON."""

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "images": [image_to_base64(image_path)],
        "stream": False
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        response.raise_for_status()
        raw = response.json().get("response", "")
        start = raw.find("{")
        end = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])
        return parsed, raw
    except Exception as e:
        print(f"  Ollama error for {image_path}: {e}")
        return None, None

def already_processed(conn, case_id):
    row = conn.execute("SELECT 1 FROM cases WHERE case_id = ?", (case_id,)).fetchone()
    return row is not None

def main():
    conn = sqlite3.connect(DB_PATH)
    init_db(conn)
    model = load_feature_extractor()

    total = 0
    skipped = 0

    for folder_name, grade in GRADE_FOLDERS.items():
        folder_path = Path(DATASET_DIR) / folder_name
        if not folder_path.exists():
            print(f"Folder not found: {folder_path}")
            continue

        images = list(folder_path.glob("*.jpg")) + list(folder_path.glob("*.png")) + list(folder_path.glob("*.jpeg"))
        images = images[:MAX_PER_GRADE]
        print(f"\n{folder_name}: processing {len(images)} images")

        for img_path in images:
            case_id = f"{folder_name}_{img_path.stem}"

            if already_processed(conn, case_id):
                print(f"  Skipping {img_path.name} (already done)")
                skipped += 1
                continue

            print(f"  Processing {img_path.name}...")

            source = "kaggle"
            name_lower = img_path.name.lower()
            if "m1" in name_lower or "mendeley1" in name_lower:
                source = "mendeley_m1"
            elif "m2" in name_lower or "mendeley2" in name_lower:
                source = "mendeley_m2"

            metadata, raw = get_metadata_from_ollama(img_path, grade)
            vec = extract_features(model, img_path)
            vec_blob = vec.astype(np.float32).tobytes()

            if metadata:
                conn.execute("""
                    INSERT OR REPLACE INTO cases
                    (case_id, image_path, kl_grade, dataset_source,
                     osteophyte_severity, joint_space_narrowing, subchondral_sclerosis,
                     bone_texture, affected_compartment, overall_findings, raw_metadata)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    case_id, str(img_path), grade, source,
                    metadata.get("osteophyte_severity"),
                    metadata.get("joint_space_narrowing"),
                    metadata.get("subchondral_sclerosis"),
                    metadata.get("bone_texture"),
                    metadata.get("affected_compartment"),
                    metadata.get("overall_findings"),
                    raw
                ))
            else:
                conn.execute("""
                    INSERT OR REPLACE INTO cases
                    (case_id, image_path, kl_grade, dataset_source, raw_metadata)
                    VALUES (?, ?, ?, ?, ?)
                """, (case_id, str(img_path), grade, source, "ollama_failed"))

            conn.execute("""
                INSERT OR REPLACE INTO features (case_id, feature_vector)
                VALUES (?, ?)
            """, (case_id, vec_blob))

            conn.commit()
            total += 1
            print(f"  Done {img_path.name} ({total} processed so far)")

    conn.close()
    print(f"\nFinished. Processed: {total}, Skipped: {skipped}")

main()

/opt/miniconda3/envs/capstone/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(



Grade_0: processing 25 images
  Processing augmented_m1_9207905L_aug2.png...
  Done augmented_m1_9207905L_aug2.png (1 processed so far)
  Processing augmented_m1_9556110R_aug3.png...
  Done augmented_m1_9556110R_aug3.png (2 processed so far)
  Processing augmented_m1_9126497L.png...
  Done augmented_m1_9126497L.png (3 processed so far)
  Processing augmented_m1_9106510L_aug2.png...
  Done augmented_m1_9106510L_aug2.png (4 processed so far)
  Processing augmented_m1_9837324R_aug1.png...
  Done augmented_m1_9837324R_aug1.png (5 processed so far)
  Processing augmented_m1_9619227L_aug1.png...
  Done augmented_m1_9619227L_aug1.png (6 processed so far)
  Processing augmented_m1_9217512L_aug2.png...
  Done augmented_m1_9217512L_aug2.png (7 processed so far)
  Processing augmented_m1_9528158R.png...
  Done augmented_m1_9528158R.png (8 processed so far)
  Processing generated_generated_01802.png...
  Done generated_generated_01802.png (9 processed so far)
  Processing augmented_m1_9747379R_au

In [2]:
import sqlite3
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np

DB_PATH = "kneevision_final.db"
MODEL_PATH = "best_model_b4.pt"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

def load_feature_extractor():
    model = models.efficientnet_b4(weights=None)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    state = checkpoint.get("model_state_dict", checkpoint)
    
    # strip backbone. prefix, skip classifier
    new_state = {}
    for k, v in state.items():
        if k.startswith("backbone.classifier"):
            continue
        if k.startswith("backbone."):
            new_state[k[len("backbone."):]] = v
    
    model.load_state_dict(new_state, strict=False)
    
    # replace classifier with identity to get feature vectors
    model.classifier = nn.Identity()
    model.eval()
    model.to(DEVICE)
    return model

def extract_features(model, image_path):
    transform = transforms.Compose([
        transforms.Resize((380, 380)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        vec = model(tensor).squeeze().cpu().numpy()
    return vec

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def retrieve_similar(image_path, model=None, top_n=5):
    if model is None:
        model = load_feature_extractor()

    query_vec = extract_features(model, image_path)

    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("""
        SELECT f.case_id, f.feature_vector,
               c.image_path, c.kl_grade, c.dataset_source,
               c.osteophyte_severity, c.joint_space_narrowing,
               c.subchondral_sclerosis, c.bone_texture,
               c.affected_compartment, c.overall_findings
        FROM features f
        JOIN cases c ON f.case_id = c.case_id
    """).fetchall()
    conn.close()

    scored = []
    for row in rows:
        case_id = row[0]
        vec = np.frombuffer(row[1], dtype=np.float32)
        sim = cosine_similarity(query_vec, vec)
        scored.append({
            "case_id": case_id,
            "similarity": round(float(sim), 4),
            "image_path": row[2],
            "kl_grade": row[3],
            "dataset_source": row[4],
            "osteophyte_severity": row[5],
            "joint_space_narrowing": row[6],
            "subchondral_sclerosis": row[7],
            "bone_texture": row[8],
            "affected_compartment": row[9],
            "overall_findings": row[10],
        })

    scored.sort(key=lambda x: x["similarity"], reverse=True)
    return scored[:top_n]


if __name__ == "__main__":
    results = retrieve_similar("augmented_kg_2347_2_aug2.png", top_n=5)
    for r in results:
        print(f"\nCase ID    : {r['case_id']}")
        print(f"Similarity : {r['similarity']}")
        print(f"KL Grade   : {r['kl_grade']}")
        print(f"Osteophytes: {r['osteophyte_severity']}")
        print(f"JSN        : {r['joint_space_narrowing']}")
        print(f"Sclerosis  : {r['subchondral_sclerosis']}")
        print(f"Texture    : {r['bone_texture']}")
        print(f"Compartment: {r['affected_compartment']}")
        print(f"Findings   : {r['overall_findings']}")


Case ID    : Grade_4_augmented_kg_2284_4_aug2
Similarity : 0.8133
KL Grade   : 4
Osteophytes: None
JSN        : None
Sclerosis  : None
Texture    : None
Compartment: None
Findings   : None

Case ID    : Grade_4_generated_generated_09292
Similarity : 0.7993
KL Grade   : 4
Osteophytes: mild
JSN        : mild
Sclerosis  : mild
Texture    : slightly irregular
Compartment: lateral
Findings   : The X-ray shows mild osteophyte and subchondral scclisekse in the lateral compartment, with a slight irregularity of the bone texture.

Case ID    : Grade_4_generated_generated_03973
Similarity : 0.7976
KL Grade   : 4
Osteophytes: mild
JSN        : mild
Sclerosis  : mild
Texture    : slightly irregular
Compartment: lateral
Findings   : The X-ray shows a Kellgren-Lawrence Grade 4 knee joint with osteophyte and joint space narrowing, subchondral scclsoe and a slightly irregular bone texture. The affected compartment is the lateral part of the joint.

Case ID    : Grade_4_generated_generated_06189
Simil

In [3]:
import torch
checkpoint = torch.load("best_model_b4.pt", map_location="cpu")
if isinstance(checkpoint, dict):
    print(checkpoint.keys())
    # if it has model_state_dict
    state = checkpoint.get("model_state_dict", checkpoint)
    keys = list(state.keys())
    print(keys[:10])
else:
    print(type(checkpoint))

odict_keys(['backbone.features.0.0.weight', 'backbone.features.0.1.weight', 'backbone.features.0.1.bias', 'backbone.features.0.1.running_mean', 'backbone.features.0.1.running_var', 'backbone.features.0.1.num_batches_tracked', 'backbone.features.1.0.block.0.0.weight', 'backbone.features.1.0.block.0.1.weight', 'backbone.features.1.0.block.0.1.bias', 'backbone.features.1.0.block.0.1.running_mean', 'backbone.features.1.0.block.0.1.running_var', 'backbone.features.1.0.block.0.1.num_batches_tracked', 'backbone.features.1.0.block.1.fc1.weight', 'backbone.features.1.0.block.1.fc1.bias', 'backbone.features.1.0.block.1.fc2.weight', 'backbone.features.1.0.block.1.fc2.bias', 'backbone.features.1.0.block.2.0.weight', 'backbone.features.1.0.block.2.1.weight', 'backbone.features.1.0.block.2.1.bias', 'backbone.features.1.0.block.2.1.running_mean', 'backbone.features.1.0.block.2.1.running_var', 'backbone.features.1.0.block.2.1.num_batches_tracked', 'backbone.features.1.1.block.0.0.weight', 'backbone.fe